# Extract the CSV

In [11]:
import os
import pandas as pd

def load_selected_snr(base_path="../data/SNR_data"):
    # Only load specific SNR files
    selected = [10, 20, 30, 50, 100]
    snr_dict = {}

    for val in selected:
        filename = f"SNR_{val}.csv"
        filepath = os.path.join(base_path, filename)
        if os.path.exists(filepath):
            snr_dict[val] = pd.read_csv(filepath)
            print(f"Loaded {filename}")
        else:
            print(f"File {filename} not found, skipping.")
    return snr_dict

if __name__ == "__main__":
    snr_data = load_selected_snr()
    # Example: preview SNR_20
    if 20 in snr_data:
        print(snr_data[20].head())


Loaded SNR_10.csv
Loaded SNR_20.csv
Loaded SNR_30.csv
Loaded SNR_50.csv
Loaded SNR_100.csv
          b         p     PERIOD  DB_OVER_DT  RS_OVER_A  NUM_TRANSITS  \
0  0.279201  0.017500  33.722826    0.011629   0.016825            46   
1  0.327023  0.018523  50.040323    0.017992   0.014303            31   
2  0.148853  0.018829  27.700893    0.016856   0.018057            56   
3  0.082522  0.026888  35.255682    0.015597   0.016018            44   
4  0.332793  0.022835  48.476562    0.018834   0.016735            32   

   snr_sqr_value  stellar_density             koi  
0      19.834739         3.492411  KOI_SNR20_Row1  
1      20.158069         2.581883  KOI_SNR20_Row2  
2      19.818403         4.187062  KOI_SNR20_Row3  
3      19.827251         3.703082  KOI_SNR20_Row4  
4      19.882412         1.717547  KOI_SNR20_Row5  


In [12]:
import numpy as np

def transit_duration_arcsin(df,
                             period_col="PERIOD",
                             rs_over_a_col="RS_OVER_A",
                             inclination_deg=90.0):
    """
    Transit duration using arcsin formula (Kipping 2010 style).
    Assumes circular orbit.
    
    Parameters
    ----------
    df : pandas.DataFrame
    period_col : str
        Column for orbital period (days).
    rs_over_a_col : str
        Column for R*/a.
    inclination_deg : float
        Inclination in degrees (default=90 = edge-on).
    
    Returns
    -------
    T : np.ndarray
        Transit duration in same units as PERIOD (e.g. days).
    """
    P = df[period_col].to_numpy()
    rs_over_a = df[rs_over_a_col].to_numpy()
    
    i = np.radians(inclination_deg)
    cosi = np.cos(i)
    sini = np.sin(i)
    
    numerator = np.sqrt(1.0 - (1.0/rs_over_a)**2 * cosi**2)
    denominator = (1.0/rs_over_a) * sini
    
    T = (P / np.pi) * np.arcsin(numerator / denominator)
    return T


In [14]:
import os
import numpy as np
import pandas as pd

# ---------- geometry & SNR ----------
def transit_duration_edgeon(df, period_col="PERIOD", rs_over_a_col="RS_OVER_A"):
    """
    Transit duration (days) assuming edge-on, circular orbit:
      T = (P/pi) * arcsin(Rs/a)
    """
    P = df[period_col].to_numpy()              # days
    rs_over_a = df[rs_over_a_col].to_numpy()   # R*/a
    return (P / np.pi) * np.arcsin(np.clip(rs_over_a, 0.0, 1.0))

def snr_per_transit_1min(df, sigma0,
                         period_col="PERIOD", rs_over_a_col="RS_OVER_A", p_col="p"):
    """
    SNR = (delta / sigma0) * sqrt(N_points), with 1-minute cadence.
    delta = p^2, N_points = T_days * 1440.
    """
    T_days = transit_duration_edgeon(df, period_col, rs_over_a_col)
    N_points = T_days * 1440.0
    delta = df[p_col].to_numpy() ** 2
    return (delta / sigma0) * np.sqrt(np.maximum(N_points, 0.0))

# ---------- run for the four groups ----------
def summarize_snr_groups(base_path="../data/SNR_data", sigma0=1e-4):
    """
    Returns a DataFrame with median and 16th/84th percentiles of SNR per transit
    for SNR_10, SNR_20, SNR_30, SNR_50.
    """
    snr_levels = [10, 20, 30, 50, 100]
    stats = []

    for level in snr_levels:
        csv_path = os.path.join(base_path, f"SNR_{level}.csv")
        if not os.path.exists(csv_path):
            print(f"Warning: {csv_path} not found, skipping.")
            continue

        df = pd.read_csv(csv_path)
        snr_vals = snr_per_transit_1min(df, sigma0=sigma0)

        median = float(np.median(snr_vals))
        p16 = float(np.percentile(snr_vals, 16))
        p84 = float(np.percentile(snr_vals, 84))

        stats.append({
            "SNR_level": level,
            "median": median,
            "p16": p16,
            "p84": p84,
            "formatted": f"{median:.3f} ({p16:.3f}–{p84:.3f})"
        })

    out = pd.DataFrame(stats).sort_values("SNR_level").reset_index(drop=True)
    return out

if __name__ == "__main__":
    # Set your noise per 1-min sample (e.g., 100 ppm = 1e-4)
    sigma0 = 2e-4
    results = summarize_snr_groups(base_path="../data/SNR_data", sigma0=sigma0)
    print(results[["SNR_level", "formatted"]].to_string(index=False))
    # If useful, also save:
    # results.to_csv("snr_per_transit_stats.csv", index=False)


 SNR_level               formatted
        10   13.581 (7.425–32.707)
        20  21.308 (10.551–56.950)
        30  32.270 (14.255–85.226)
        50  31.551 (16.083–81.769)
       100 54.959 (24.157–134.248)
